# Deep & Two-Tower Recommenders

Companion notebook for the [Deep & Two-Tower lesson](https://ml-viz-ruby.vercel.app/courses/recommender-systems/03-deep-and-two-tower).

**The idea in one sentence.** Encode users and items with **two separate towers**
into one shared vector space, so scoring is a **dot product** — which means item
embeddings can be precomputed offline and retrieval over billions of items becomes
a fast nearest-neighbour search.

Why this architecture dominates industrial recsys:

- **Decoupled towers** → item vectors are computed once, offline; only the user
  tower runs at request time.
- **In-batch-negatives contrastive loss** → every other item in the batch is a
  free negative, so training needs only positive (user, item) pairs.
- **Retrieval → ranking funnel** → a cheap dot-product retrieves a shortlist, then
  a heavy model re-scores only those few.

We build all three from scratch, **validate the contrastive loss against SciPy and
the retrieval against the exact top-k**, then cover the gotchas. Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)

## 1 — Two-tower scoring is a dot product in a shared space

The user tower and item tower each output a d-dim vector; the score is their dot product. Because
the towers are separate, **item embeddings can be precomputed once** and reused for every user.

In [ ]:
d = 8
n_items = 1000
item_emb = rng.normal(size=(n_items, d))                 # precomputed OFFLINE
item_emb /= np.linalg.norm(item_emb, axis=1, keepdims=True)

def user_tower(history_item_ids):
    """A toy user tower: average the embeddings of items the user engaged with."""
    u = item_emb[history_item_ids].mean(0)
    return u / np.linalg.norm(u)

u = user_tower([3, 17, 42])
scores = item_emb @ u                                    # one dot product per item
print('top-5 retrieved item ids:', np.argsort(-scores)[:5])
print('the user history items score high:', sorted(scores[[3,17,42]].round(2), reverse=True))

## 2 — In-batch-negatives contrastive loss

For a batch of B positive (user, item) pairs, each user's positive is its paired item and the
negatives are the *other* items in the batch. The loss is a softmax cross-entropy over the B×B
score matrix with the diagonal as the targets — B(B-1) negatives for free.

In [ ]:
def in_batch_loss(U, V):
    """U: (B,d) user embeddings, V: (B,d) their positive item embeddings."""
    logits = U @ V.T                                     # (B,B): row i vs all items in batch
    logits -= logits.max(1, keepdims=True)
    p = np.exp(logits) / np.exp(logits).sum(1, keepdims=True)
    B = len(U)
    return -np.mean(np.log(p[np.arange(B), np.arange(B)] + 1e-9))

B = 6
# aligned pairs (each user embedding ~ its positive item) -> low loss
Vp = rng.normal(size=(B, d)); Vp /= np.linalg.norm(Vp, axis=1, keepdims=True)
Up = Vp + 0.05 * rng.normal(size=(B, d))                 # users near their positives
# random (mis-aligned) pairs -> high loss
Ur = rng.normal(size=(B, d))
print(f'loss, aligned pairs: {in_batch_loss(Up, Vp):.3f}  (should be low)')
print(f'loss, random  pairs: {in_batch_loss(Ur, Vp):.3f}  (should be higher)')

### Validate: the in-batch loss is cross-entropy, and aligned beats random

The in-batch-negatives loss is exactly softmax **cross-entropy** with the batch
diagonal as labels — we confirm it against `scipy.special.log_softmax`, and check
that aligned (user≈positive) pairs really do score a lower loss than random ones.

In [ ]:
from scipy.special import log_softmax as sp_log_softmax

def loss_via_scipy(U, V):
    logits = U @ V.T
    B = len(U)
    return -sp_log_softmax(logits, axis=1)[np.arange(B), np.arange(B)].mean()

print(f'ours   (aligned): {in_batch_loss(Up, Vp):.6f}')
print(f'scipy  (aligned): {loss_via_scipy(Up, Vp):.6f}')
assert np.isclose(in_batch_loss(Up, Vp), loss_via_scipy(Up, Vp)), 'must equal softmax cross-entropy'
assert in_batch_loss(Up, Vp) < in_batch_loss(Ur, Vp), 'aligned pairs must score lower loss'
print('\n✅ in-batch-negatives loss IS cross-entropy; aligned pairs beat random')

## 3 — The retrieval → ranking funnel

Retrieval cheaply shortlists candidates by dot product (recall). A heavier ranker then re-scores
only that shortlist with richer cross-features (precision). We simulate the two stages.

In [ ]:
def retrieve(u, item_emb, k):
    return np.argsort(-(item_emb @ u))[:k]               # cheap, scans all items

def ranker_score(u, item_ids, item_emb):
    # a heavier (here, nonlinear) scorer applied ONLY to the shortlist
    v = item_emb[item_ids]
    return (v @ u) ** 2 + 0.1 * v.sum(1)                 # toy cross-feature score

shortlist = retrieve(u, item_emb, k=50)                  # billions -> 50 (cheap)
ranked = shortlist[np.argsort(-ranker_score(u, shortlist, item_emb))][:10]
print('retrieval shortlist (first 8):', shortlist[:8])
print('final top-10 after ranking:   ', ranked)
print('ranking only re-scored 50 items, not all', n_items)

### Validate: the funnel returns a subset, ranked, without scanning everything

The retrieval → ranking split is only useful if ranking touches *far* fewer items
than retrieval. We confirm the final list is a subset of the retrieved shortlist
(ranking never invents new items) and that it re-scored only the shortlist.

In [ ]:
assert set(ranked.tolist()) <= set(shortlist.tolist()), 'ranking must only reorder the shortlist'
assert len(shortlist) < n_items, 'retrieval must shrink the candidate set'
print(f'retrieval scanned all {n_items} items -> shortlist of {len(shortlist)}')
print(f'ranking re-scored only {len(shortlist)} items -> final {len(ranked)}')
print('\n✅ the funnel: cheap dot-product retrieval, then heavy ranking on a tiny shortlist')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **must be a dot product** | any score that mixes user & item non-linearly kills precomputation/ANN — that's the ranker's role |
| **in-batch negatives are biased** | popular items appear as negatives more often → correct with log-Q / mixed negatives |
| **embedding normalization** | unnormalized dot products conflate direction and magnitude; normalize for cosine |
| **retrieval recall ceiling** | anything the retriever misses, the ranker can never recover |
| **stale item embeddings** | precomputed offline → must be refreshed as items/behaviour drift |

Demo: only the dot-product form scales — a single-tower MLP would need one forward
pass per candidate.

In [ ]:
# Why a SINGLE tower (concatenate user+item, then MLP) can't scale to retrieval:
# its score is not a dot product, so you cannot precompute item vectors or use a
# nearest-neighbour index — you'd have to run the MLP for every (user, item) pair.
import time
n_big = 200_000
big_items = rng.normal(size=(n_big, d)); big_items /= np.linalg.norm(big_items, axis=1, keepdims=True)
t0 = time.time(); _ = np.argsort(-(big_items @ u))[:10]; dot_ms = (time.time()-t0)*1000
print(f'two-tower dot-product retrieval over {n_big:,} items: {dot_ms:.1f} ms')
print(f'a single-tower MLP would run {n_big:,} forward passes PER REQUEST instead.')
print('The dot-product factorization is exactly what makes ANN retrieval possible.')

## ✏️ Your turn

**Exercise.** Implement `two_tower_score(u, v)` (the dot product of a user and item embedding) and
`retrieve_topk(u, item_emb, k)` returning the ids of the `k` highest-scoring items. This is exactly
the nearest-neighbor retrieval an ANN index accelerates in production.

In [ ]:
def two_tower_score(u, v):
    # TODO(you): score = dot product of the user and item embeddings
    return ...

def retrieve_topk(u, item_emb, k):
    # TODO(you): return the ids of the k items with the highest score against u
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert np.isclose(two_tower_score(u, item_emb[7]), u @ item_emb[7])
top = retrieve_topk(u, item_emb, 5)
assert list(top) == list(np.argsort(-(item_emb @ u))[:5])
# the top retrieved item scores at least as high as any other
assert two_tower_score(u, item_emb[top[0]]) >= two_tower_score(u, item_emb[123])
print('\u2713 two-tower scoring and top-k retrieval are correct')

<details>
<summary>Solution</summary>

```python
def two_tower_score(u, v):
    return u @ v

def retrieve_topk(u, item_emb, k):
    return np.argsort(-(item_emb @ u))[:k]
```

Because scoring is a dot product against precomputed item vectors, retrieval is exactly a
nearest-neighbor search — which an ANN index (HNSW/IVF) turns from O(items) into milliseconds at
billion-item scale.

</details>

## Key takeaways

- **Two towers → dot-product scoring → precomputable item vectors.** This is what
  lets retrieval scale to billions of items via nearest-neighbour search.
- **In-batch negatives** turn positive-only data into a contrastive task — the
  loss is plain softmax cross-entropy (we matched SciPy).
- **Retrieval → ranking funnel:** a cheap dot product narrows billions to a
  shortlist, then a heavy scorer re-ranks only those.
- **The price of decoupling:** the towers can't model user–item cross-features
  directly — that's the ranking stage's job.